# 01 YAMNet setup exploration

In [10]:
import sys
sys.path.append('..')

import importlib
import pandas as pd
import numpy as np

import spectra.models.classifier as classifier
importlib.reload(classifier)

from spectra.models.classifier import classify_wav
from spectra.models.category_mapping import map_to_category

Loading YAMNet model...


E0000 00:00:1787682339.294500   69954 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


YAMNet model loaded successfully.


In [3]:
print("Cargando YAMNet desde TF-Hub...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
print("Modelo cargado correctamente ✅")

Cargando YAMNet desde TF-Hub...


E0000 00:00:1787580946.955798   28552 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Modelo cargado correctamente ✅


In [4]:
sample_rate = 16000
duration_seconds = 1
zero_waveform = np.zeros(sample_rate * duration_seconds, dtype=np.float32)

scores, embeddings, spectrogram = yamnet_model(zero_waveform)

print(f"Scores shape: {scores.shape}")
print(f"Embeddings shape: {embeddings.shape}")
print(f"Spectrogram shape: {spectrogram.shape}")

Scores shape: (2, 521)
Embeddings shape: (2, 1024)
Spectrogram shape: (144, 64)


In [5]:
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = pd.read_csv(class_map_path)['display_name'].tolist()

print(f"Total de clases: {len(class_names)}")
print(class_names[:10])

Total de clases: 521
['Speech', 'Child speech, kid speaking', 'Conversation', 'Narration, monologue', 'Babbling', 'Speech synthesizer', 'Shout', 'Bellow', 'Whoop', 'Yell']


In [6]:
import soundfile as sf

# Generate a simple test tone (440 Hz = A note, "beep"-like)
sample_rate = 16000
duration = 3  # seconds
frequency = 440

t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
test_tone = 0.5 * np.sin(2 * np.pi * frequency * t)
test_tone = test_tone.astype(np.float32)

# Save as .wav
output_path = "../raw_data/test_tone.wav"
sf.write(output_path, test_tone, sample_rate)

print(f"File saved at: {output_path}")

File saved at: ../raw_data/test_tone.wav


## 02 Esc50 Evaluation


In [2]:
import pandas as pd

esc50_meta = pd.read_csv("../raw_data/esc50_dataset/meta/esc50.csv")

print(f"Total clips: {len(esc50_meta)}")
print(f"\nColumns: {esc50_meta.columns.tolist()}")
print(f"\nUnique categories: {esc50_meta['category'].nunique()}")
print(esc50_meta.head())

Total clips: 2000

Columns: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take']

Unique categories: 50
            filename  fold  target        category  esc10  src_file take
0   1-100032-A-0.wav     1       0             dog   True    100032    A
1  1-100038-A-14.wav     1      14  chirping_birds  False    100038    A
2  1-100210-A-36.wav     1      36  vacuum_cleaner  False    100210    A
3  1-100210-B-36.wav     1      36  vacuum_cleaner  False    100210    B
4  1-101296-A-19.wav     1      19    thunderstorm  False    101296    A


In [15]:
print(sorted(esc50_meta['category'].unique()))


['airplane', 'breathing', 'brushing_teeth', 'can_opening', 'car_horn', 'cat', 'chainsaw', 'chirping_birds', 'church_bells', 'clapping', 'clock_alarm', 'clock_tick', 'coughing', 'cow', 'crackling_fire', 'crickets', 'crow', 'crying_baby', 'dog', 'door_wood_creaks', 'door_wood_knock', 'drinking_sipping', 'engine', 'fireworks', 'footsteps', 'frog', 'glass_breaking', 'hand_saw', 'helicopter', 'hen', 'insects', 'keyboard_typing', 'laughing', 'mouse_click', 'pig', 'pouring_water', 'rain', 'rooster', 'sea_waves', 'sheep', 'siren', 'sneezing', 'snoring', 'thunderstorm', 'toilet_flush', 'train', 'vacuum_cleaner', 'washing_machine', 'water_drops', 'wind']


In [3]:
ESC50_TO_CATEGORY = {
    # Alert
    "siren": "Alert",
    "clock_alarm": "Alert",
    "fireworks": "Alert",
    "glass_breaking": "Alert",

    # Human
    "clapping": "Human",
    "laughing": "Human",
    "coughing": "Human",
    "sneezing": "Human",
    "crying_baby": "Human",
    "breathing": "Human",
    "footsteps": "Human",
    "snoring": "Human",
    "drinking_sipping": "Human",
    "brushing_teeth": "Human",

    # Animal
    "dog": "Animal",
    "cat": "Animal",
    "cow": "Animal",
    "pig": "Animal",
    "hen": "Animal",
    "rooster": "Animal",
    "crow": "Animal",
    "frog": "Animal",
    "chirping_birds": "Animal",
    "insects": "Animal",
    "sheep": "Animal",
    "crickets": "Animal",

    # Vehicle
    "car_horn": "Vehicle",
    "engine": "Vehicle",
    "train": "Vehicle",
    "airplane": "Vehicle",
    "helicopter": "Vehicle",
    "chainsaw": "Vehicle",

    # Music: ESC-50 no tiene categoría musical real, se omite (no keys map here)

    # Background (ambient / non-actionable sounds)
    "rain": "Background",
    "sea_waves": "Background",
    "thunderstorm": "Background",
    "wind": "Background",
    "crackling_fire": "Background",
    "water_drops": "Background",
    "pouring_water": "Background",
    "church_bells": "Background",
    "clock_tick": "Background",
    "keyboard_typing": "Background",
    "mouse_click": "Background",
    "door_wood_knock": "Background",
    "door_wood_creaks": "Background",
    "can_opening": "Background",
    "washing_machine": "Background",
    "vacuum_cleaner": "Background",
    "toilet_flush": "Background",
    "hand_saw": "Background",
}

In [4]:
# Filter dataset to only categories we've mapped (excludes anything ambiguous/unmapped)
esc50_meta['true_category'] = esc50_meta['category'].map(ESC50_TO_CATEGORY)
esc50_filtered = esc50_meta.dropna(subset=['true_category']).copy()

print(f"Filtered dataset: {len(esc50_filtered)} clips (from {len(esc50_meta)} total)")
print(esc50_filtered['true_category'].value_counts())

Filtered dataset: 2000 clips (from 2000 total)
true_category
Background    720
Animal        480
Human         400
Vehicle       240
Alert         160
Name: count, dtype: int64


# 03 - ESC-50 Evaluation & Mapping

In [ ]:
import time

# Take a balanced sample: up to 15 clips per category, for fast iteration
SAMPLE_PER_CATEGORY = 15

sampled_dfs = []
for category in esc50_filtered['true_category'].unique():
    category_subset = esc50_filtered[esc50_filtered['true_category'] == category]
    n_samples = min(len(category_subset), SAMPLE_PER_CATEGORY)
    sampled_dfs.append(category_subset.sample(n_samples, random_state=42))

esc50_sample = pd.concat(sampled_dfs).reset_index(drop=True)

print(f"Sample size: {len(esc50_sample)} clips")
print(esc50_sample['true_category'].value_counts())

Sample size: 75 clips
true_category
Animal        15
Background    15
Human         15
Alert         15
Vehicle       15
Name: count, dtype: int64


In [11]:
results = []
start_time = time.time()

for idx, row in esc50_sample.iterrows():
    filepath = f"/home/pablo/code/spectra/raw_data/esc50_dataset/audio/{row['filename']}"
    print(filepath)

    true_category = row['true_category']

    clip_start = time.time()
    try:
        predictions = classify_wav(filepath, top_n=1)
        
        top_class, top_confidence = predictions[0]
        predicted_category = map_to_category(top_class, top_confidence)
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")
        continue
    clip_latency = time.time() - clip_start

    results.append({
        "filename": row['filename'],
        "true_category": true_category,
        "predicted_category": predicted_category,
        "yamnet_top_class": top_class,
        "yamnet_confidence": top_confidence,
        "correct": true_category == predicted_category,
        "latency_seconds": clip_latency
    })

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(esc50_sample)} clips...")

total_time = time.time() - start_time
print(f"\nDone. Total time: {total_time:.1f}s ({total_time/len(esc50_sample):.2f}s per clip avg)")

results_df = pd.DataFrame(results)

/home/pablo/code/spectra/raw_data/esc50_dataset/audio/1-69641-A-3.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/5-198278-B-7.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/5-156026-C-4.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/3-70962-B-4.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/5-177614-A-5.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/2-80844-A-13.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/3-233151-A-2.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/4-250869-B-2.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/1-73585-A-7.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/5-215172-A-13.wav
Processed 10/75 clips...
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/1-17970-A-4.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/3-71964-C-4.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/5-212454-A-0.wav
/home/pablo/code/spectra/raw_data/esc50_dataset/audio/4-

In [12]:
print(classify_wav)
print(map_to_category)

<function classify_wav at 0x7626af451940>
<function map_to_category at 0x7626af4a07c0>


In [18]:
# Apply map_to_category and handle whatever format it returns
mapped_results = results_df.apply(
    lambda row: map_to_category(
        row["yamnet_top_class"], row["yamnet_confidence"]
    ),
    axis=1,
)

# If it returns a single category string
if isinstance(mapped_results.iloc[0], str):
    results_df["predicted_category"] = mapped_results
# If it returns a tuple or list
else:
    results_df["predicted_category"] = [res[0] for res in mapped_results]

# Check match between ground truth and predicted category
results_df["correct"] = (
    results_df["true_category"] == results_df["predicted_category"]
)

# Calculate global metrics
accuracy = results_df["correct"].mean() * 100
avg_latency = results_df["latency_seconds"].mean()

print(f"Global Accuracy: {accuracy:.2f}%")
print(f"Average Latency: {avg_latency:.4f} seconds per clip")

# Accuracy breakdown per category
category_acc = (
    results_df.groupby("true_category")["correct"].mean() * 100
).reset_index()
category_acc.columns = ["Category", "Accuracy (%)"]
print("\nAccuracy per Category:")
print(category_acc)

Global Accuracy: 54.67%
Average Latency: 0.1339 seconds per clip

Accuracy per Category:
     Category  Accuracy (%)
0       Alert     33.333333
1      Animal     46.666667
2  Background     86.666667
3       Human     40.000000
4     Vehicle     66.666667


Calibrate Noise Gate / Threshold

In [19]:
# Threshold calibration loop for noise gate
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
calibration_results = []

for thresh in thresholds:
    # Map with varying threshold
    temp_mapped = results_df.apply(
        lambda row: map_to_category(
            row["yamnet_top_class"], row["yamnet_confidence"], threshold=thresh
        ),
        axis=1,
    )

    # Extract predicted category based on returned format
    if isinstance(temp_mapped.iloc[0], str):
        pred_cats = temp_mapped
    else:
        pred_cats = [res[0] for res in temp_mapped]

    is_correct = results_df["true_category"] == pred_cats
    acc = is_correct.mean() * 100

    calibration_results.append({"Threshold": thresh, "Accuracy (%)": round(acc, 2)})

# Display results table
thresh_df = pd.DataFrame(calibration_results)
print(thresh_df)

   Threshold  Accuracy (%)
0        0.1         56.00
1        0.2         56.00
2        0.3         54.67
3        0.4         49.33
4        0.5         45.33
